In [1]:
from copy import deepcopy

import numpy as np

from conf.behavior_cloning.diffusion.five_demos.default import config
from rlbench.action_modes.arm_action_modes import BimanualEndEffectorPoseViaIK
from tapas_gmm.encoder.encoder import ObservationEncoderConfig
from tapas_gmm.env.rlbench import RLBenchEnvironment, RLBenchEnvironmentConfig
from tapas_gmm.policy.diffusion import DiffusionPolicy
from tapas_gmm.utils.select_gpu import device

2026-07-17 20:47:20.717 | INFO     |  Running on cpu


In [2]:
joint_config = deepcopy(config.policy)
leader_config = deepcopy(config.policy)
follower_config = deepcopy(config.policy)

for policy_config in (joint_config, leader_config, follower_config):
    policy_config.obs_dim = 35
    policy_config.horizon = 16
    policy_config.n_obs_steps = 2
    policy_config.n_action_steps = 8
    policy_config.training = None
    policy_config.obs_encoder = ObservationEncoderConfig(
        ee_pose=True,
        object_poses=True,
    )
    policy_config.unet.down_dims = (64, 128, 256)

joint_config.action_dim = 16
joint_config.unet.input_dim = 16
joint_config.unet.global_cond_dim = 35 * 2

leader_config.action_dim = 8
leader_config.arm = "left"
leader_config.unet.input_dim = 8
leader_config.unet.global_cond_dim = 35 * 2

follower_config.action_dim = 8
follower_config.arm = "right"
follower_config.condition_on_arm = "left"
follower_config.unet.input_dim = 8
follower_config.unet.global_cond_dim = 35 * 2 + 16 * 8

In [3]:
joint_policy = DiffusionPolicy(joint_config).to(device)
leader_policy = DiffusionPolicy(leader_config).to(device)
follower_policy = DiffusionPolicy(follower_config).to(device)

joint_policy.from_disk("../outputs/bimanual_diffusion_policy.pt")
leader_policy.from_disk("../outputs/diffusion_leader_left.pt")
follower_policy.from_disk("../outputs/diffusion_follower_right.pt")

joint_policy.eval()
leader_policy.eval()
follower_policy.eval()

2026-07-17 20:47:23.853 | INFO     |  Initializing DiffusionPolicy:
2026-07-17 20:47:23.864 | INFO     |    Initializing Policy:
2026-07-17 20:47:24.029 | INFO     |    number of parameters: 5525968
2026-07-17 20:47:24.037 | INFO     |    No encoder config provided. Using None.
None
2026-07-17 20:47:24.169 | INFO     |    number of parameters: 5522376
None
2026-07-17 20:47:24.279 | INFO     |    number of parameters: 5981128
None


DiffusionPolicy(
  (model): ConditionalUnet1D(
    (mid_modules): ModuleList(
      (0-1): 2 x ConditionalResidualBlock1D(
        (blocks): ModuleList(
          (0-1): 2 x Conv1dBlock(
            (block): Sequential(
              (0): Conv1d(256, 256, kernel_size=(5,), stride=(1,), padding=(2,))
              (1): GroupNorm(8, 256, eps=1e-05, affine=True)
              (2): Mish()
            )
          )
        )
        (cond_encoder): Sequential(
          (0): Mish()
          (1): Linear(in_features=454, out_features=512, bias=True)
          (2): Rearrange('batch t -> batch t 1')
        )
        (residual_conv): Identity()
      )
    )
    (diffusion_step_encoder): Sequential(
      (0): SinusoidalPosEmb()
      (1): Linear(in_features=256, out_features=1024, bias=True)
      (2): Mish()
      (3): Linear(in_features=1024, out_features=256, bias=True)
    )
    (up_modules): ModuleList(
      (0): ModuleList(
        (0): ConditionalResidualBlock1D(
          (blocks): M

In [4]:
tapas_env = RLBenchEnvironment(
    RLBenchEnvironmentConfig(
        action_mode=BimanualEndEffectorPoseViaIK,
        robot_setup="dual_panda",
        task="BimanualDualPushButtons",
        cameras=tuple(),
        camera_pose={},
        image_size=(128, 128),
        static=False,
        headless=False,
        scale_action=False,
        delay_gripper=False,
        gripper_plot=False,
        absolute_action_mode=True,
        action_frame="world",
    )
)

In [5]:
def predict_joint(obs):
    trajectory, _ = joint_policy.predict(obs)
    return np.concatenate((trajectory.ee, trajectory.gripper), axis=-1)


def predict_leader_follower(obs):
    leader_trajectory, leader_info = leader_policy.predict(obs)
    follower_trajectory, _ = follower_policy.predict(
        obs,
        condition=leader_info["action_pred"],
    )
    return np.concatenate(
        (
            leader_trajectory.ee,
            follower_trajectory.ee,
            leader_trajectory.gripper[:, None],
            follower_trajectory.gripper[:, None],
        ),
        axis=-1,
    )

In [6]:
def run_episode(architecture, max_steps=200):
    obs = tapas_env.reset()
    joint_policy.reset_episode(tapas_env)
    leader_policy.reset_episode(tapas_env)
    follower_policy.reset_episode(tapas_env)

    total_reward = 0
    step = 0
    done = False

    while step < max_steps and not done:
        if architecture == "joint":
            actions = predict_joint(obs)
        else:
            actions = predict_leader_follower(obs)

        for action in actions:
            obs, reward, done, _ = tapas_env.step(action)
            total_reward += reward
            step += 1
            if done or obs is None or step >= max_steps:
                break

    print("architecture:", architecture)
    print("steps:", step)
    print("total_reward:", total_reward)

In [7]:
#run_episode("joint")

In [8]:
run_episode("leader_follower")

[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.


2026-07-17 20:47:44.018 | INFO     |  Action [ 0.2096376  -0.15511021  1.06455779 -0.49334347  0.86868745  0.04453563
  0.00330455  1.          0.          0.28558594  0.12976564  1.1451683
 -0.03736511  0.97590023  0.2052239   0.0640762   1.          0.        ]
2026-07-17 20:47:44.171 | INFO     |  Skipping invalid action [ 0.2096376  -0.15511021  1.06455779 -0.49334347  0.86868745  0.04453563
  0.00330455  1.          0.          0.28558594  0.12976564  1.1451683
 -0.03736511  0.97590023  0.2052239   0.0640762   1.          0.        ].
2026-07-17 20:47:44.176 | INFO     |  Can't execute noop either. Just haning in there...
architecture: leader_follower
steps: 1
total_reward: 0


In [9]:
tapas_env.close()

[CoppeliaSim:loadinfo]   done.
